# 01 — ETL: painel de cidades e snapshots
Cliente fino do pacote `supply_experiments`. Toda a lógica (filtros da base de
pedidos, normalização de cidades, estatísticas de elegibilidade) vive no pacote
— **uma única implementação**, testada, em vez das 4 cópias divergentes anteriores.

In [ ]:
%pip install /Workspace/Shared/supply_experiments/supply_experiments-1.0.2-py3-none-any.whl --quiet --force-reinstall --no-deps
dbutils.library.restartPython()

In [ ]:
from supply_experiments.spark_io import load_city_panel, TABLES
from supply_experiments.design.spillover import EligibilityCriteria, eligible_cities

# ---- parâmetros -----------------------------------------------------------
dbutils.widgets.text("target_env", "sandbox")     # sandbox | prod
dbutils.widgets.text("date_start", "2025-09-01")
dbutils.widgets.text("date_end",   "2026-06-30")

TARGET_ENV = dbutils.widgets.get("target_env")
DATE_START = dbutils.widgets.get("date_start")
DATE_END   = dbutils.widgets.get("date_end")
tables = TABLES[TARGET_ENV]

In [ ]:
# painel diário (GMV + num/den de ruptura) e stats por cidade — 1 chamada
panel, stats = load_city_panel(spark, DATE_START, DATE_END)
print(f"Painel: {len(panel.cities)} cidades x {len(panel.index)} dias")
stats.head(10)

In [ ]:
# elegibilidade: filtros de volume + NOVOS critérios de qualidade de série
crit = EligibilityCriteria(
    min_nonzero_days=92,
    min_avg_daily_gmv=10_000.0,
    max_zero_run_days=7,   # buracos de dados longos => fora
    max_cv=1.5,            # séries erráticas => fora
)
elig = eligible_cities(stats, crit)
print(f"{len(elig)} cidades elegíveis de {len(stats)}")
display(elig.sort_values("avg_daily_gmv", ascending=False).head(30))

In [ ]:
# persiste snapshot de elegibilidade p/ os notebooks de design
elig_sdf = spark.createDataFrame(elig.reset_index())
elig_sdf.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{tables['registry'].rsplit('.',1)[0]}.eligible_cities_snapshot")